In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/dhotepatil00@gmail.com/regis-healthcare/1_setup/utility

In [0]:
# %run /Workspace/Users/gayatrijoshi663@gmail.com/regis-healthcare/1_setup/utility

In [0]:
print(bronze_schema,silver_schema,gold_schema) 

In [0]:
dbutils.widgets.text("catalog","regis_healthcare","catalog")
dbutils.widgets.text("data_source","employees","data_source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

#### Silver Processing

In [0]:
df_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source};")
display(df_bronze)
print(df_bronze.count())

In [0]:
# schema check
print(df_bronze.count())
df_bronze.printSchema()

In [0]:
df_bronze.columns

In [0]:
# drop duplicate
df_silver = df_bronze.dropDuplicates()
print(df_silver.count())

In [0]:
df_silver = df_silver.withColumn(
    "employee_id",
    F.trim(F.col("employee_id"))
).withColumn(
    "first_name",
    F.trim(F.col("first_name"))
).withColumn(
    "last_name",
    F.trim(F.col("last_name"))
).withColumn(
    "job_title",
    F.trim(F.col("job_title"))
).withColumn(
    "facility_id",
    F.trim(F.col("facility_id"))
).withColumn(
    "phone",
    F.trim(F.col("phone"))
).withColumn(
    "email",
    F.trim(F.col("email"))
).withColumn(
    "hire_date",
    F.trim(F.col("hire_date"))
).withColumn(
    "employment_type",
    F.trim(F.col("employment_type"))
).withColumn(
    "status",
    F.trim(F.col("status"))
).withColumn(
    "salary",
    F.trim(F.col("salary"))
).withColumn(
    "created_at",
    F.trim(F.col("created_at"))
)

In [0]:
# null records count 
from pyspark.sql.functions import col,count,when
null_count = df_silver.select([count(when(col(c).isNull(),c)).alias(c)for c in df_silver.columns
                               ])
display(null_count)

#### Cleaning data in table

In [0]:
# employee_id

check = df_silver.filter(~col("employee_id").rlike("^EMP"))
display(check)
df_silver = df_silver.withColumn("employee_id",when(~col("employee_id").rlike("^EMP"),None).otherwise(col("employee_id")))
display(df_silver)


In [0]:
# first_name
from pyspark.sql.functions import col,when,trim,initcap

df_silver = df_silver.withColumn("first_name",initcap(trim(col("first_name"))))

dup = df_silver.groupBy("first_name").count().filter(col("count")>1)
display(dup)

In [0]:
# last_name
from pyspark.sql.functions import col,when,trim,initcap

df_silver = df_silver.withColumn("first_name",initcap(trim(col("first_name"))))

dup = df_silver.groupBy("first_name").count().filter(col("count")>1)
display(dup)

In [0]:
# job_title

from pyspark.sql.functions import when, lit, col, lower, trim,upper
df_silver = df_silver.withColumn("job_title",upper(trim(col("job_title"))))
dup = df_silver.groupBy("job_title").count().filter(col("count")>1)
# display(dup)
invalid_values = ["NAN",
"N/A",
"#N/A",
"NONE",
"UNKNOWN",
"NULL",""]

df_silver = df_silver.withColumn("job_title",
    when(
        col("job_title").isNull() | upper(trim(col("job_title"))).isin(invalid_values),
        lit("UNKNOWN")
    ).otherwise(trim(col("job_title")))
)

# display(df_silver)
dup = df_silver.groupBy("job_title").count().filter(col("count")>1)
display(dup)

In [0]:
# facility_id

from pyspark.sql.functions import col,when
df_filt = df_silver.filter(~col("facility_id").rlike("^FAC"))

df_silver = df_silver.withColumn(
    "facility_id",
    when(
        (col("facility_id").isNull()) | (~col("facility_id").rlike("^FAC")),
        "0"
    ).otherwise(col("facility_id"))
)

display(df_filt)
display(df_silver)

In [0]:
# phone
dup = df_silver.filter(~col("phone").rlike("^[0-9]{10}$")
)
# display(dup)
dp = dup.groupBy("phone").count().filter(col("count")>1)
# display(dp)

reple_xx = {"(02) 0000" : "0",
"ABCDEFG" : "0",
"1234567890123456" : "0",
"+61 0" : "0",
"000-000-0000" : "0",
"123" : "0",
"0000000000":"0",
"":"0"}

from pyspark.sql.functions import  col

df_silver = df_silver.replace(reple_xx,subset=["phone"])
dp = df_silver.groupBy("phone").count().filter(col("count")>1)
display(dp)
dup = df_silver.filter(~col("phone").rlike("^[0-9]{10}$")
)
display(df_silver)

In [0]:
# email

dup = df_silver.groupBy("email").count().filter(col("count")>1)
# display(dup)


from pyspark.sql.functions import col, when, lower, trim, lit

invalid_values = [
"@nodomain.com",
"N/A",
"user@.com",
"user@@example.com",
"user@",
"NaN",
  "",
"#N/A",
"notanemail",
"NULL",
"NONE",
"null",
"UNKNOWN",
"hunter.long164@icloud.com"]

df_silver = df_silver.withColumn(
    "email",
    when(
        col("email").isNull() | trim(col("email")).isin(invalid_values),
        lit("unknown")
    ).otherwise(trim(col("email")))
)

# display(df_silver)
dup = df_silver.groupBy("email").count().filter(col("count")>1)
display(dup)

In [0]:
# hire_date

from pyspark.sql.functions import to_timestamp
dup= df_silver.groupBy("hire_date").count().filter(col("count")>1)
# display(dup)

filt_repl = ["00/00/0000",
"not-a-date",
"2026-02-30",
"32/13/2026",
"2030-01-01",
"9999-99-99"]

df_silver = df_silver.withColumn("hire_date",when(col("hire_date").isin(filt_repl),None).otherwise(col("hire_date")))
dup= df_silver.groupBy("hire_date").count().filter(col("count")>1)
# display(dup)
df_silver = df_silver.withColumn("hire_date",to_timestamp(col("hire_date"),"yyyy-MM-dd HH:mm:ss"))
display(df_silver)

In [0]:
# employment_type

from pyspark.sql.functions import col,when
df_silver = df_silver.withColumn("employment_type",upper(trim(col("employment_type"))))
dup= df_silver.groupBy("employment_type").count().filter(col("count")>1)
display(dup)

filt_repl = ["N/A",
"NAN",
"null",
"#N/A",
"NULL",
"NONE",
"UNKNOWN",
""
]

df_silver = df_silver.withColumn("employment_type",when(col("employment_type").isin(filt_repl),"UNKNOWN").otherwise(col("employment_type")))

df_silver = df_silver.fillna({"employment_type":"UNKNOWN"})

dup= df_silver.groupBy("employment_type").count().filter(col("count")>1)
display(dup)
# display(df_silver)

In [0]:
# status
from pyspark.sql.functions import col, when, upper, trim, lit ,initcap

invalid_values = ["null", "nan", "n/a", "#n/a", "none", ""]

df_silver = df_silver.withColumn("status",
    when(col("status").isNull() |
        upper(trim(col("status"))).isin(invalid_values),lit("not active")
    ).otherwise(upper(trim(col("status"))))
)
display(df_silver)
dup= df_silver.groupBy("status").count()
display(dup)

In [0]:
# salary
from pyspark.sql.functions import col, abs

df_silver = df_silver.withColumn(
    "salary",
    abs(col("salary").cast("double"))
)
df_invalid = df_silver.filter(col("salary").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
display(df_invalid)
display(df_silver)

In [0]:
# created_at
dup= df_silver.groupBy("created_at").count().filter(col("count")>1)
display(dup)

df_silver = df_silver.withColumn("created_at",to_timestamp(col("created_at"),"yyyy-MM-dd HH:mm:ss"))
display(df_silver)

#### silver table load

In [0]:
df_silver.write\
    .format("delta")\
        .option("delta.enableChangeDataFeed","true")\
            .option("mergeSchema","true")\
                .option("overwriteSchema","true")\
            .mode("overwrite")\
               .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

dt = spark.sql(f"select * from {catalog}.{silver_schema}.{data_source};")
print(dt.count())
display(dt)

In [0]:
# load to s3
df_silver.write.format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
    .mode("overwrite")\
    .partitionBy("current_date")\
    .save(f"s3://regis-healthcare/silver-clean-data/{data_source}/")